# NeSy-Mamba v7 Training Analysis

**Key result:** 81.7% val accuracy (273K params, 20 epochs on ProofWriter)

**Remaining issue:** Slots are decorative — 4/7 dead, 3 alive but ALL at identical value (~0.5518)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Parsed from Kaggle logs ──────────────────────────────────────
epochs = list(range(1, 21))

# End-of-epoch summaries
train_acc  = [0.526, 0.559, 0.570, 0.657, 0.680, 0.755, 0.818, 0.825, 0.831, 0.839,
              0.847, 0.853, 0.860, 0.868, 0.875, 0.880, 0.887, 0.891, 0.893, 0.895]
val_acc    = [0.557, 0.556, 0.604, 0.658, 0.675, 0.805, 0.813, 0.811, 0.814, 0.817,
              0.817, 0.815, 0.815, 0.815, 0.812, 0.813, 0.809, 0.812, 0.808, 0.809]
task_loss  = [0.6273, 0.5982, 0.5874, 0.5362, 0.5145, 0.4192, 0.3207, 0.3114, 0.3026, 0.2918,
              0.2807, 0.2702, 0.2601, 0.2481, 0.2372, 0.2260, 0.2164, 0.2089, 0.2038, 0.2016]
lr_vals    = [0.000333, 0.000667, 0.001000, 0.000992, 0.000970, 0.000933, 0.000883, 0.000821,
              0.000750, 0.000671, 0.000587, 0.000500, 0.000413, 0.000329, 0.000250, 0.000179,
              0.000117, 0.000067, 0.000030, 0.000008]

# Slot diagnostics
gate_mean  = [0.258, 0.261, 0.258, 0.262, 0.265, 0.263, 0.260, 0.260, 0.261, 0.260,
              0.261, 0.263, 0.261, 0.265, 0.263, 0.266, 0.265, 0.260, 0.260, 0.266]
collapsed  = [0.571]*20  # constant throughout
frac_above = [0.449, 0.458, 0.451, 0.455, 0.462, 0.460, 0.450, 0.457, 0.453, 0.454,
              0.452, 0.452, 0.452, 0.458, 0.453, 0.461, 0.462, 0.455, 0.454, 0.461]
slot_entropy= [0.357, 0.356, 0.351, 0.361, 0.360, 0.359, 0.357, 0.350, 0.356, 0.351,
               0.353, 0.354, 0.349, 0.353, 0.352, 0.355, 0.354, 0.346, 0.346, 0.357]

# Per-class accuracy
acc_true   = [0.495, 0.315, 0.553, 0.602, 0.649, 0.852, 0.898, 0.877, 0.872, 0.913,
              0.878, 0.886, 0.864, 0.879, 0.860, 0.871, 0.847, 0.865, 0.858, 0.859]
acc_false  = [0.630, 0.847, 0.665, 0.726, 0.707, 0.748, 0.710, 0.731, 0.743, 0.702,
              0.742, 0.730, 0.757, 0.740, 0.754, 0.743, 0.762, 0.748, 0.749, 0.748]

# Per-depth accuracy at best epoch (10)
depths_best = {0: 0.9307, 1: 0.6906, 2: 0.7627, 3: 0.7744, 4: 0.6017, 5: 0.5000}

# v6 comparison (from previous run, at best epoch ~8)
v6_val_acc = 0.660
v6_depths  = {0: 0.679, 1: 0.670, 2: 0.645, 3: 0.595}
v6_collapsed = 0.714  # 5/7 slots dead

print(f"v7 Best val acc: {max(val_acc):.4f} (epoch {val_acc.index(max(val_acc))+1})")
print(f"v7 Best w/ threshold sweep: 0.8180 (thresh=0.55)")
print(f"v6 Best val acc: {v6_val_acc:.4f}")
print(f"Improvement: +{max(val_acc) - v6_val_acc:.1%}")

In [ ]:
# ── Figure 1: Accuracy Curves ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1a: Train vs Val accuracy
ax = axes[0]
ax.plot(epochs, train_acc, 'b-o', markersize=4, label='Train')
ax.plot(epochs, val_acc, 'r-s', markersize=4, label='Val')
ax.axhline(y=0.544, color='gray', ls='--', alpha=0.5, label='Majority class')
ax.axhline(y=v6_val_acc, color='orange', ls='--', alpha=0.7, label=f'v6 best ({v6_val_acc:.3f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('v7 Training Curves')
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 0.95)

# 1b: Task loss + LR
ax = axes[1]
ax.plot(epochs, task_loss, 'g-o', markersize=4, label='Task Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Task Loss', color='g')
ax.tick_params(axis='y', labelcolor='g')
ax2 = ax.twinx()
ax2.plot(epochs, [lr*1000 for lr in lr_vals], 'purple', ls='--', alpha=0.7, label='LR (×1e-3)')
ax2.set_ylabel('LR (×1e-3)', color='purple')
ax2.tick_params(axis='y', labelcolor='purple')
ax.set_title('Loss & Cosine LR Schedule')
ax.grid(True, alpha=0.3)

# 1c: Per-class accuracy
ax = axes[2]
ax.plot(epochs, acc_true, 'b-o', markersize=4, label='Acc True')
ax.plot(epochs, acc_false, 'r-s', markersize=4, label='Acc False')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy (True vs False)')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0.3, 1.0)

plt.tight_layout()
plt.savefig('v7_accuracy_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Per-Depth Accuracy (v6 vs v7) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2a: v6 vs v7 bar chart
ax = axes[0]
depth_labels = ['d0', 'd1', 'd2', 'd3']
v6_vals = [v6_depths[i] for i in range(4)]
v7_vals = [depths_best[i] for i in range(4)]
x = np.arange(len(depth_labels))
w = 0.35
bars1 = ax.bar(x - w/2, v6_vals, w, label='v6 (best 66%)', color='orange', alpha=0.8)
bars2 = ax.bar(x + w/2, v7_vals, w, label='v7 (best 81.7%)', color='steelblue', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(depth_labels)
ax.set_ylabel('Val Accuracy'); ax.set_title('Per-Depth: v6 vs v7')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0.4, 1.0)
# Add value labels
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=8)

# 2b: v7 all depths
ax = axes[1]
all_depths = sorted(depths_best.keys())
all_vals = [depths_best[d] for d in all_depths]
n_examples = [7606, 4651, 2824, 2154, 118, 18]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0', '#795548']
bars = ax.bar(all_depths, all_vals, color=colors, alpha=0.8)
ax.axhline(y=0.544, color='gray', ls='--', alpha=0.5, label='Majority baseline')
ax.set_xlabel('Proof Depth'); ax.set_ylabel('Val Accuracy')
ax.set_title('v7 Accuracy by Proof Depth (best epoch 10)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0.0, 1.0)
for bar, n in zip(bars, n_examples):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.1%}\n(n={n})', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('v7_depth_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: SLOT HEALTH — the problem ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 3a: Gate mean + collapsed fraction
ax = axes[0]
ax.plot(epochs, gate_mean, 'b-o', markersize=4, label='gate_mean')
ax.plot(epochs, collapsed, 'r-s', markersize=5, label='frac_collapsed')
ax.set_xlabel('Epoch'); ax.set_ylabel('Value')
ax.set_title('Slot Health: Gate Mean & Collapse')
ax.legend(); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
ax.annotate('4/7 slots dead\n(constant!)', xy=(10, 0.571), fontsize=11,
            color='red', fontweight='bold',
            xytext=(12, 0.75), arrowprops=dict(arrowstyle='->', color='red'))
ax.annotate('gate_mean ≈ 0.26\n(flat line!)', xy=(10, 0.26), fontsize=11,
            color='blue', fontweight='bold',
            xytext=(12, 0.15), arrowprops=dict(arrowstyle='->', color='blue'))

# 3b: Slot entropy
ax = axes[1]
ax.plot(epochs, slot_entropy, 'g-o', markersize=4)
ax.set_xlabel('Epoch'); ax.set_ylabel('Slot Entropy')
ax.set_title('Slot Entropy (higher = more diverse)')
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1)
ax.axhline(y=np.log(7), color='gray', ls='--', alpha=0.5, label=f'Max entropy (ln 7 = {np.log(7):.2f})')
ax.annotate('Entropy flat at ~0.35\n(no diversification)', xy=(10, 0.35),
            fontsize=11, color='darkgreen', fontweight='bold',
            xytext=(3, 0.7), arrowprops=dict(arrowstyle='->', color='green'))
ax.legend()

# 3c: v6 vs v7 slot comparison
ax = axes[2]
metrics = ['collapsed', 'gate_mean', 'frac>0.5', 'entropy']
v6_slot = [0.714, 0.095, 0.05, 0.15]  # approx v6 values
v7_slot = [0.571, 0.260, 0.454, 0.351]
x = np.arange(len(metrics))
w = 0.35
ax.bar(x - w/2, v6_slot, w, label='v6', color='orange', alpha=0.8)
ax.bar(x + w/2, v7_slot, w, label='v7', color='steelblue', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_title('Slot Health: v6 vs v7')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('v7_slot_health.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Findings

### Accuracy: Excellent
| Metric | v6 | v7 | Change |
|--------|------|------|--------|
| Val Accuracy | 66.0% | **81.7%** | **+15.7pp** |
| Thresh-tuned | — | **81.8%** | — |
| Train Acc (final) | 71.4% | **89.5%** | **+18.1pp** |
| depth-0 | 67.9% | **93.1%** | +25.2pp |
| depth-1 | 67.0% | **69.1%** | +2.1pp |
| depth-2 | 64.5% | **76.3%** | +11.8pp |
| depth-3 | 59.5% | **77.4%** | +17.9pp |

### Slots: Still Broken
| Metric | v6 | v7 | Issue |
|--------|------|------|-------|
| Collapsed | 71.4% (5/7) | **57.1% (4/7)** | 1 slot woke up, still 4 dead |
| gate_mean | 0.095 | **0.260** | Higher but flat — never changes |
| frac>0.5 | ~5% | **45%** | 3 slots at ~0.55 (all identical!) |
| entropy | ~0.15 | **0.35** | Higher but static |

### Self-Explanation Demo: The Smoking Gun
All 5 examples show the **exact same slot values**: Rule_1=0.5518, Rule_5=0.5518, Rule_4=0.5518.

**The slots encode NO example-specific information.** They're constant offsets that the model has learned to ignore.

## Root Cause Diagnosis

### Why slots are constant across examples

The slot gate equation is: $s_k(t) = \max(s_k(t-1),\ \sigma(W_s \cdot h(t) + u_k \cdot s_k(t-1) + b_k))$

The **monotonic max** means once a slot activates, it can NEVER deactivate. With:
- `b_s = -1.5` → $\sigma(-1.5) \approx 0.18$ as starting point
- `u_diag = +0.01` → very slight self-excitation

The slot starts at ~0.18 and the max operation means after just a few timesteps, ALL slots reach ~0.55 regardless of input content because:

1. **The bias term dominates** — $b_k$ is input-independent, so every example gets the same initial activation
2. **Monotonic max is irreversible** — once `s(t)>0.5`, it's locked in, even if later evidence says "this rule doesn't apply"
3. **W_s projection is too weak** — `gain=0.1` Xavier init means the input-dependent signal $W_s \cdot h(t)$ is tiny compared to the bias

### The fundamental architectural issue

The **monotonic max** gate was designed to model accumulating evidence (like a proof building up), but in practice:
- It prevents the model from learning **negative evidence** ("this rule definitely does NOT apply")
- All slots converge to the same value because the bias + recurrence dominate over input-specific W_s projections
- The entropy loss can't fix this because it pushes for diversity, but the max operation prevents any slot from going DOWN

### Why accuracy is still great
The Mamba backbone (2 layers, d=128) is perfectly capable of binary classification on its own. The slot gate is a parallel module that the model learns to mostly ignore. The answer is computed from `h_final` which carries all the sequence information regardless of slot values.

## Proposed v8 Fixes: Making Slots Actually Work

### Option A: Replace monotonic max with EMA-based gate
Replace the irreversible `max(s_prev, new_val)` with a learnable exponential moving average:
$$s_k(t) = \alpha_k \cdot s_k(t-1) + (1 - \alpha_k) \cdot \sigma(W_s \cdot h(t) + b_k)$$

where $\alpha_k \in (0,1)$ is a per-slot learned smoothing factor.

**Pros:** Slots can both rise AND fall based on input, allowing differentiation.

### Option B: Use FINAL timestep only (no recurrence)
$$s_k = \sigma(W_s \cdot h_{\text{final}} + b_k)$$

Just project the final hidden state to K slot values.

**Pros:** Input-dependent by construction. Simplest fix.
**Cons:** Loses temporal accumulation story.

### Option C: Gumbel-softmax slot selection
Instead of sigmoid per slot, use Gumbel-softmax to select a SPARSE subset of active slots:
$$s = \text{GumbelSoftmax}(W_s \cdot h_{\text{final}} / \tau)$$

**Pros:** Forces differentiation, naturally sparse.

### Option D: Increase W_s gain + reduce bias effect
Less invasive: increase `gain=0.1` to `gain=1.0` for W_s, use `b_s = 0.0`, and rely on input signal:
$$s_k(t) = \max(s_k(t-1),\ \sigma(W_s \cdot h(t) + u_k \cdot s_k(t-1)))$$

Remove bias entirely so slots MUST depend on input content.

### Recommendation: Option A (EMA gate) is the best balance of:
1. Interpretability (still has temporal trace)
2. Differentiation (slots can deactivate)
3. Minimal code change (replace 1 line in forward())

In [ ]:
# ── Summary Table ────────────────────────────────────────────────
print("="*70)
print("  NeSy-Mamba v7 FINAL RESULTS")
print("="*70)
print(f"  Model: d=128, layers=2, slots=7, params=273K")
print(f"  Dataset: ProofWriter (118K train, 17K val, depths 0-5)")
print(f"")
print(f"  Val Accuracy:        81.72%  (best epoch 10)")
print(f"  Threshold-tuned:     81.80%  (threshold=0.55)")
print(f"  Train Accuracy:      89.5%   (epoch 20)")
print(f"  Logic Fidelity:      63.3%")
print(f"  Slot Fidelity:       54.9%")
print(f"")
print(f"  Depth-0 (easy):      93.1%")
print(f"  Depth-1:             69.1%")
print(f"  Depth-2:             76.3%")
print(f"  Depth-3 (hard):      77.4%")
print(f"")
print(f"  SLOT STATUS:   ❌ 4/7 collapsed, 3 alive but identical")
print(f"  ACCURACY:      ✅ +15.7pp over v6")
print(f"  INTERPRETABLE: ❌ Slots don't vary with input")
print(f"="*70)
print(f"\n  NEXT: v8 with EMA gate to fix slot differentiation")